<a href="https://colab.research.google.com/github/datweb07/repo-nosql/blob/main/baitap2_nosql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 28.4 MB/s eta 0:00:00


In [2]:
from datetime import datetime
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
from google.colab import userdata

uri = userdata.get('MONGODB_URI')

client = MongoClient(uri, server_api=ServerApi('1'))

try:
    client.admin.command('ping')
    print("Kết nối MongoDB thành công!")
except Exception as e:
    print(e)

Kết nối MongoDB thành công!


In [3]:
db = client['DB_BookStore']

collection_book = db['book']
print("đã tạo thành công db và collection!\n")

đã tạo thành công db và collection!



In [4]:
sample_books = [
    {"_id": 1, "title": "Sách A", "category": "IT", "price": 100, "tags": ["code", "web"], "stock": 10},
    {"_id": 2, "title": "Sách B", "category": "IT", "price": 150, "tags": ["code", "AI"], "stock": 4},
    {"_id": 3, "title": "Sách C", "category": "Kinh Te", "price": 150, "tags": ["money", "startup"], "stock": 20},
    {"_id": 4, "title": "Sách D", "category": "Kinh Te", "price": 80, "tags": ["startup"], "stock": 3},
    {"_id": 5, "title": "Sách E", "category": "Van Hoc", "price": 90, "tags": ["truyen", "tinh cam"], "stock": 50}
]

collection_book.insert_many(sample_books)
print("tạo dữ liệu thành công!")

tạo dữ liệu thành công!


# Tính tổng số tồn kho và đếm số sách của từng thể loại

In [6]:
q1 = [
    {
        "$group": {
            '_id': '$category',
            'TongTonKho' : {
                '$sum': '$stock'
            },
            'SoDauSach' : {
                '$count': {}
            }
        }
    },
    {
         '$sort': {
        'TongTonKho' : -1
    }
    }
]

for i in collection_book.aggregate(q1):
  print(i)

{'_id': 'Van Hoc', 'TongTonKho': 50, 'SoDauSach': 1}
{'_id': 'Kinh Te', 'TongTonKho': 23, 'SoDauSach': 2}
{'_id': 'IT', 'TongTonKho': 14, 'SoDauSach': 2}


# tìm các cuốn sách có giá đắt nhất trong cửa hàng

In [8]:
q2 = [
    {
        '$group':{
            '_id': 'null',
            'MaxPrice': {
                '$max': '$price'
            },
            'AllBooks': {
                '$push': '$$ROOT'
            }
        }
    },
    {
        '$unwind' : '$AllBooks'
    },
    {
        '$match' : {
            '$expr' : {
                '$eq' : [
                    '$MaxPrice' , '$AllBooks.price'
                ]
            }
        }
    },
    {
        '$project' : {
            '_id': 0,
            'Title': '$AllBooks.title',
            'Category': '$AllBooks.category',
            'Price': '$AllBooks.price'
        }
    }
]

for i in collection_book.aggregate(q2):
  print(i)



{'Title': 'Sách B', 'Category': 'IT', 'Price': 150}
{'Title': 'Sách C', 'Category': 'Kinh Te', 'Price': 150}


# tăng giá của tất cả sách 'Kinh Te' lên thêm 20
# thêm thẻ 'bestseller' vào mảng tags của những sách này (không thể trùng lăp)

In [10]:
filter_query = {
    'category': 'Kinh Te'
}

update_operation = {
    '$inc': {
        'price': 20
    },
    '$addToSet': {
        'tags': 'bestseller'
    }
}

collection_book.update_many(filter_query, update_operation)
print("update thành công")

update thành công
